# Final Causal Fused Stream Smoke

Ce notebook reprend le script `final_causal_fused_stream_smoke.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Smoke test causal frame-by-frame: frame t utilise seulement frame t et historique <= t.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Run a causal frame-by-frame final fused risk smoke test on a raw video.
- Commande de reproduction referencee : causal final fused raw-video stream smoke.
- Artefacts controles : Causal frame-by-frame final fused raw-video stream smoke exists. (`runs/exp_051_causal_final_fused_stream_smoke/causal_fused_stream_config.json`).
- Le script ecrit ou lit des artefacts experimentaux dans `runs/`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_causal_fused_stream_smoke.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import time
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image

from final_fused_inference_demo import (
    BASE_MODELS,
    FINAL_SCORE_COLS,
    apply_final_formulas,
    crop_transform,
    load_crop_checkpoint,
    load_sequence_checkpoint,
    resolve,
    safe_person_crop,
    selected_crop_arches,
    write_contact_sheet,
)
from ml_pipeline import YOLO_POSE_WEIGHTS, load_dataset, load_json, pose_rows_for_results, write_json, zone_polygon
from sequence_inference_demo import build_sequences


## Fonction `choose_device`

Cette cellule definit `choose_device`. Elle prepare une partie du script.

In [ ]:
def choose_device(requested):
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(requested)


## Fonction `update_ema`

Cette cellule definit `update_ema`. Elle prepare une partie du script.

In [ ]:
def update_ema(previous, value, alpha):
    if previous is None:
        return float(value)
    return float(alpha * value + (1.0 - alpha) * previous)


## Fonction `predict_crop_frame`

Cette cellule definit `predict_crop_frame`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict_crop_frame(model, frame, pose_row, transform, device):
    crop = safe_person_crop(frame, pose_row)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    tensor = transform(Image.fromarray(rgb)).unsqueeze(0).to(device)
    return float(torch.sigmoid(model(tensor).view(-1))[0].detach().cpu())


## Fonction `predict_sequence_frame`

Cette cellule definit `predict_sequence_frame`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict_sequence_frame(seed_runtime, pose_history, feature_cols, device):
    output = {
        "repeat_seed": seed_runtime["seed"],
        "attention_risk": seed_runtime["attention_ema"],
        "ppe_risk": seed_runtime["ppe_ema"],
    }
    pose_df = pd.DataFrame(pose_history)
    sequence_cache = {}
    for base, item in seed_runtime["sequence"].items():
        model = item["model"]
        payload = item["payload"]
        seq_len = int(payload["seq_len"])
        if seq_len not in sequence_cache:
            sequence_cache[seq_len] = build_sequences(
                pose_df,
                feature_cols,
                seed_runtime["mean"],
                seed_runtime["std"],
                seq_len,
            )
        X = sequence_cache[seq_len]
        xb = torch.from_numpy(X[-1:]).to(device)
        probs = torch.sigmoid(model(xb)).detach().cpu().numpy()[0]
        horizons = [float(h) for h in payload["horizons"]]
        h_idx = horizons.index(1.0) if 1.0 in horizons else int(np.argmin(np.abs(np.asarray(horizons) - 1.0)))
        danger = float(probs[h_idx])
        output[f"sequence_{base}"] = danger
        meta_payload = item["meta"]
        meta_input = pd.DataFrame(
            {
                "danger_risk_1.0s": [danger],
                "attention_risk": [seed_runtime["attention_ema"]],
                "ppe_risk": [seed_runtime["ppe_ema"]],
            }
        )
        output[f"learned_meta_{base}"] = float(meta_payload["model"].predict_proba(meta_input[meta_payload["meta_cols"]])[:, 1][0])
    scored = apply_final_formulas(pd.DataFrame([output]))
    return scored.iloc[0].to_dict()


## Fonction `load_runtime`

Cette cellule definit `load_runtime`. Elle prepare une partie du script.

In [ ]:
def load_runtime(args, device):
    fusion_run = resolve(args.fusion_run)
    feature_run = resolve(args.sequence_feature_run)
    feature_payload = load_json(feature_run / "features" / "sequence_feature_columns.json")
    transform = crop_transform(args.image_size)
    seeds = []
    for seed in args.seeds:
        attention_arch, blouse_arch = selected_crop_arches(fusion_run, seed)
        attention_model, _ = load_crop_checkpoint(fusion_run / "models" / f"attention_seed{seed}_{attention_arch}.pt", device)
        blouse_model, _ = load_crop_checkpoint(fusion_run / "models" / f"blouse_seed{seed}_{blouse_arch}.pt", device)
        normalizer = np.load(fusion_run / "features" / f"sequence_normalizer_seed{seed}.npz")
        seed_runtime = {
            "seed": int(seed),
            "attention_arch": attention_arch,
            "blouse_arch": blouse_arch,
            "attention_model": attention_model,
            "blouse_model": blouse_model,
            "attention_ema": None,
            "ppe_ema": None,
            "mean": normalizer["mean"].astype(np.float32),
            "std": normalizer["std"].astype(np.float32),
            "sequence": {},
        }
        for base in BASE_MODELS:
            model, payload = load_sequence_checkpoint(fusion_run / "models" / f"seed{seed}_{base}.pt", device)
            meta_payload = joblib.load(fusion_run / "models" / f"meta_seed{seed}_{base}.joblib")
            seed_runtime["sequence"][base] = {"model": model, "payload": payload, "meta": meta_payload}
        seeds.append(seed_runtime)
    return {
        "fusion_run": fusion_run,
        "feature_run": feature_run,
        "feature_cols": feature_payload["feature_columns"],
        "transform": transform,
        "seeds": seeds,
    }


## Fonction `summarize_latency`

Cette cellule definit `summarize_latency`. Elle prepare une partie du script.

In [ ]:
def summarize_latency(pred):
    cols = ["pose_latency_ms", "crop_latency_ms", "sequence_latency_ms", "total_latency_ms"]
    rows = []
    for col in cols:
        values = pred[col].to_numpy(dtype=np.float64)
        rows.append(
            {
                "component": col.replace("_latency_ms", ""),
                "mean_ms": float(np.mean(values)),
                "median_ms": float(np.median(values)),
                "p95_ms": float(np.percentile(values, 95)),
                "max_ms": float(np.max(values)),
                "estimated_fps_from_mean": float(1000.0 / max(1e-9, np.mean(values))),
            }
        )
    return pd.DataFrame(rows)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    out_dir = resolve(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_video = resolve(args.raw_video)
    device = choose_device(args.device)
    videos, _, _, zones = load_dataset()
    polygon = zone_polygon(zones)
    runtime = load_runtime(args, device)

    from ultralytics import YOLO

    pose_model = YOLO(str(YOLO_POSE_WEIGHTS))
    cap = cv2.VideoCapture(str(raw_video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_meta = {
        "video_id": args.video_id or raw_video.stem,
        "path": str(raw_video),
        "fps": fps,
        "width": width,
        "height": height,
    }

    pose_history = []
    prediction_rows = []
    crop_rows = []
    alarm_run = 0
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if args.max_frames is not None and frame_idx >= args.max_frames:
            break

        total_start = time.perf_counter()
        pose_start = time.perf_counter()
        result = pose_model.predict([frame], imgsz=args.imgsz, conf=args.pose_conf, verbose=False)[0]
        pose_row = pose_rows_for_results(video_meta, "inference", [frame_idx], [result], polygon)[0]
        pose_history.append(pose_row)
        pose_ms = (time.perf_counter() - pose_start) * 1000.0

        crop_start = time.perf_counter()
        seed_crop_values = {}
        for seed_runtime in runtime["seeds"]:
            seed = seed_runtime["seed"]
            attention_now = predict_crop_frame(seed_runtime["attention_model"], frame, pose_row, runtime["transform"], device)
            ppe_now = predict_crop_frame(seed_runtime["blouse_model"], frame, pose_row, runtime["transform"], device)
            seed_runtime["attention_ema"] = update_ema(seed_runtime["attention_ema"], attention_now, args.crop_ema_alpha)
            seed_runtime["ppe_ema"] = update_ema(seed_runtime["ppe_ema"], ppe_now, args.crop_ema_alpha)
            seed_crop_values[f"attention_risk_seed{seed}"] = seed_runtime["attention_ema"]
            seed_crop_values[f"ppe_risk_seed{seed}"] = seed_runtime["ppe_ema"]
            crop_rows.append(
                {
                    "frame": frame_idx,
                    "time_s": frame_idx / fps,
                    "seed": seed,
                    "attention_arch": seed_runtime["attention_arch"],
                    "blouse_arch": seed_runtime["blouse_arch"],
                    "attention_risk_current": attention_now,
                    "ppe_risk_current": ppe_now,
                    "attention_risk_ema": seed_runtime["attention_ema"],
                    "ppe_risk_ema": seed_runtime["ppe_ema"],
                }
            )
        crop_ms = (time.perf_counter() - crop_start) * 1000.0

        sequence_start = time.perf_counter()
        seed_score_rows = []
        for seed_runtime in runtime["seeds"]:
            seed_score_rows.append(predict_sequence_frame(seed_runtime, pose_history, runtime["feature_cols"], device))
        sequence_ms = (time.perf_counter() - sequence_start) * 1000.0

        row = {
            "video_id": video_meta["video_id"],
            "path": str(raw_video),
            "frame": frame_idx,
            "time_s": frame_idx / fps,
            "fps": fps,
            "pose_latency_ms": pose_ms,
            "crop_latency_ms": crop_ms,
            "sequence_latency_ms": sequence_ms,
        }
        row.update(seed_crop_values)
        for seed_score in seed_score_rows:
            seed = int(seed_score["repeat_seed"])
            for col in ["attention_risk", "ppe_risk", *FINAL_SCORE_COLS]:
                if col in seed_score:
                    row[f"{col}_seed{seed}"] = float(seed_score[col])
        for col in ["attention_risk", "ppe_risk", *FINAL_SCORE_COLS]:
            seed_cols = [f"{col}_seed{seed_runtime['seed']}" for seed_runtime in runtime["seeds"] if f"{col}_seed{seed_runtime['seed']}" in row]
            if seed_cols:
                row[f"{col}_mean{len(seed_cols)}"] = float(np.mean([row[c] for c in seed_cols]))

        score_col = args.score_col
        if score_col not in row:
            candidate = f"{score_col}_mean{len(runtime['seeds'])}"
            if candidate in row:
                score_col = candidate
            else:
                raise SystemExit(f"Score column not found: {args.score_col}")
        score = float(row[score_col])
        alarm_run = alarm_run + 1 if score >= args.threshold else 0
        row["score_col"] = score_col
        row["threshold"] = float(args.threshold)
        row["alarm"] = int(alarm_run >= args.persistence_frames)
        row["total_latency_ms"] = (time.perf_counter() - total_start) * 1000.0
        prediction_rows.append(row)
        frame_idx += 1

    cap.release()
    pred = pd.DataFrame(prediction_rows)
    if pred.empty:
        raise SystemExit("No frames were processed")
    crop_trace = pd.DataFrame(crop_rows)
    latency = summarize_latency(pred)
    score_col = str(pred["score_col"].iloc[0])
    pred.to_csv(out_dir / "causal_fused_stream_predictions.csv", index=False)
    crop_trace.to_csv(out_dir / "causal_crop_risk_trace.csv", index=False)
    latency.to_csv(out_dir / "causal_latency_summary.csv", index=False)
    contact_sheet = write_contact_sheet(out_dir, raw_video, pred, polygon, score_col, args.threshold)
    config = {
        "fusion_run": str(runtime["fusion_run"]),
        "sequence_feature_run": str(runtime["feature_run"]),
        "video_path": str(raw_video),
        "video_id": video_meta["video_id"],
        "device": str(device),
        "raw_video_mode": True,
        "causal_streaming_policy": "Frame t uses YOLO pose from frame t, current-frame crop CNN predictions updated by EMA, and sequence features from frames <= t only.",
        "crop_ema_alpha": float(args.crop_ema_alpha),
        "seeds": [int(item["seed"]) for item in runtime["seeds"]],
        "score_col": score_col,
        "threshold": float(args.threshold),
        "persistence_frames": int(args.persistence_frames),
        "rows": int(len(pred)),
        "max_score": float(pred[score_col].max()),
        "first_alarm_time_s": None if pred[pred["alarm"] == 1].empty else float(pred[pred["alarm"] == 1]["time_s"].iloc[0]),
        "mean_total_latency_ms": float(pred["total_latency_ms"].mean()),
        "estimated_fps_from_mean_total_latency": float(1000.0 / max(1e-9, pred["total_latency_ms"].mean())),
        "contact_sheet": None if contact_sheet is None else str(contact_sheet),
    }
    write_json(out_dir / "causal_fused_stream_config.json", config)
    write_json(out_dir / "causal_latency_summary.json", {"rows": latency.to_dict(orient="records")})
    print(out_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Run a causal frame-by-frame final fused risk smoke test on a raw video.")
    parser.add_argument("--fusion-run", default="runs/exp_020_same_split_fusion")
    parser.add_argument("--sequence-feature-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--raw-video", required=True)
    parser.add_argument("--video-id", default=None)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--seeds", nargs="+", type=int, default=[111, 222, 333])
    parser.add_argument("--score-col", default="final_learned_meta_mean")
    parser.add_argument("--threshold", type=float, default=0.48)
    parser.add_argument("--persistence-frames", type=int, default=2)
    parser.add_argument("--crop-ema-alpha", type=float, default=0.20)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--pose-conf", type=float, default=0.10)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--max-frames", type=int, default=None)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement concret du smoke test causal final
# Cette configuration reprend la video utilisee dans le run de reference.
from datetime import datetime
import sys

OUT_DIR = f"runs/exp_051_causal_final_fused_stream_smoke_notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RAW_VIDEO = r"C:\Users\ilyas\Desktop\pose recognision\captures\ilyas\unsafe\unsafe blooza good  20260512_190912.mp4"
NOTEBOOK_ARGS = [
    "--raw-video", RAW_VIDEO,
    "--out-dir", OUT_DIR,
    "--score-col", "final_learned_meta_mean_mean3",
    "--threshold", "0.48",
]

ancien_argv = sys.argv[:]
sys.argv = ["final_causal_fused_stream_smoke.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
